In [ ]:
train_dir = '/kaggle/input/sgfood-train-test/datasets/train'
test_dir = '/kaggle/input/sgfood-train-test/datasets/test'

In [ ]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [ ]:
import os
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

class EfficientNet(nn.Module):
    def __init__(self, class_no):
        super(EfficientNet, self).__init__()
        self.base = models.efficientnet_b0(pretrained=True)
        # fine-tune the last two layers
        for param in self.base.features[-2:].parameters():
            param.requires_grad = True
        self.base.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(self.base.classifier[1].in_features, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, class_no)
        )
    def forward(self, x):
        return self.base(x)

In [ ]:
from torch.utils.data import DataLoader, random_split
def load_data(train_dir, test_dir, batch_size):
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        # normalize to match with pretrained value
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])
    data = datasets.ImageFolder(train_dir, transform=transform)
    val_size = int(0.2 * len(data))
    train_size = len(data) - val_size
    train_data, val_data = random_split(data, [train_size, val_size])
    test_data = datasets.ImageFolder(test_dir, transform=transform)

    train_loader = DataLoader(train_data, batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_data, batch_size, shuffle=False, num_workers=2)

    test_loader  = DataLoader(test_data, batch_size)
    return train_loader, val_loader, test_loader

In [ ]:
import time
from torch.amp import autocast, GradScaler
def train(model, train_loader, val_loader, criterion, optimizer, epochs):
    scaler = GradScaler()
    min_val_loss = float('inf')
    patience = 2
    trigger_times = 0
    model.to(device)
    for epoch in range(epochs):
        start_time = time.time()
        model.train()
        training_loss = 0.0
        for i, l in train_loader:
            i, l = i.to(device), l.to(device)
            optimizer.zero_grad()
            # reduce precision for less mem & faster training
            with autocast(device_type='cuda'):
                output = model(i)
                loss = criterion(output, l)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            training_loss += loss.item()

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for i, l in val_loader:
                i, l = i.to(device), l.to(device)
                with autocast(device_type='cuda'):
                    outputs = model(i)
                    loss = criterion(outputs, l)
                val_loss += loss.item()
        avg_val_loss = val_loss / len(val_loader)
        epoch_time = time.time() - start_time
        print(f'epoch [{epoch+1}/{epochs}], train loss: {training_loss/len(train_loader):.4f}  | val loss: {val_loss/len(val_loader):.4f}  | '
              f'time: {epoch_time:.2f}s')

        # early stopping
        if avg_val_loss < min_val_loss:
            min_val_loss = avg_val_loss
            best_model_wts = model.state_dict()
            trigger_times = 0
        else:
            trigger_times += 1
            if trigger_times >= patience:
                print("early stopping")
                break

    model.load_state_dict(best_model_wts)
    return model


In [ ]:
def evaluate(model, test_loader, criterion):
    model.to(device)
    model.eval()
    correct = 0
    total = 0
    test_loss = 0.0

    with torch.no_grad():
        for i, l in test_loader:
            i, l = i.to(device), l.to(device)
            output = model(i)
            loss = criterion(output, l)
            test_loss += loss.item()

            _, predicted = torch.max(output.data, 1)
            total += l.size(0)
            correct += (predicted == l).sum().item()

    accuracy = correct * 100 / total
    avg_loss = test_loss / len(test_loader)

    print(f'test loss: {avg_loss:.4f} | test accuracy: {accuracy:.4f}%')

In [ ]:
class_no = len(os.listdir(train_dir))
train_loader, val_loader, test_loader = load_data(train_dir, test_dir, 64)
model = EfficientNet(class_no)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)
train(model, train_loader, val_loader, criterion, optimizer, 10)
evaluate(model, test_loader, criterion)

epoch [1/10], train loss: 3.1937  | val loss: 1.5747  | time: 99.27s
epoch [2/10], train loss: 1.3938  | val loss: 1.0175  | time: 98.88s
epoch [3/10], train loss: 0.9735  | val loss: 0.8921  | time: 99.07s
epoch [4/10], train loss: 0.7341  | val loss: 0.8384  | time: 98.66s
epoch [5/10], train loss: 0.5841  | val loss: 0.8270  | time: 100.15s
epoch [6/10], train loss: 0.4582  | val loss: 0.8197  | time: 100.99s
epoch [7/10], train loss: 0.3681  | val loss: 0.8439  | time: 99.48s
epoch [8/10], train loss: 0.3056  | val loss: 0.8670  | time: 99.90s
early stopping
test loss: 0.9074 | test accuracy: 77.6480%


In [ ]:
torch.save(model.state_dict(), 'efficientNet_model.pth')